# Module 02 — Dynamic Programming: Policy & Value Iteration

**Reinforcement Learning · Dakar Institute of Technology (DIT)**

When we **know the model** of the environment — the transition probabilities
$P(s'\mid s,a)$ and rewards $R(s,a,s')$ — we can compute the **optimal policy
exactly**, without any trial-and-error learning. This is **Dynamic Programming (DP)**,
and it is the theoretical foundation for everything that follows.

### The Bellman equations
For a policy $\pi$, the **state-value function** satisfies the *Bellman expectation
equation*:
$$V^\pi(s) = \sum_a \pi(a\mid s) \sum_{s'} P(s'\mid s,a)\,\big[R + \gamma V^\pi(s')\big]$$

The **optimal** value function satisfies the *Bellman optimality equation*:
$$V^*(s) = \max_a \sum_{s'} P(s'\mid s,a)\,\big[R + \gamma V^*(s')\big]$$

### Learning objectives
1. Implement **iterative policy evaluation**.
2. Implement **policy iteration** (evaluation + greedy improvement).
3. Implement **value iteration**.
4. Compare the two and **watch the optimal policy act** in the GridWorld.

We use the course `GridWorld`, which exposes the full model as
`env.P[s][a] -> [(prob, next_state, reward, done), ...]`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "rl_helpers").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from rl_helpers import (make_default_gridworld, set_seed, animate_policy,
                        plot_state_values, plot_policy)

set_seed(0)
env = make_default_gridworld(slip=0.1, gamma=0.99)
print("GridWorld (S=start, G=goal +1, H=hole -1, ##=wall):")
print(env.render_ascii())
print(f"\nStates: {env.n_states}, Actions: {env.n_actions}, gamma: {env.gamma}, slip: {env.slip}")

## 0. Inspect the model

The whole point of DP is that we have `env.P`. Let's look at one entry: from the
start state, taking action **RIGHT**, with `slip=0.1` the agent mostly goes right
but sometimes slips perpendicular.

In [ ]:
from rl_helpers import RIGHT
s0 = env.rc_to_s(0, 0)
print(f"P[start][RIGHT] (prob, next_state, reward, done):")
for prob, ns, r, done in env.P[s0][RIGHT]:
    print(f" prob={prob:.2f} -> state {ns} {env.s_to_rc(ns)} reward={r:+.2f} done={done}")

## 1. Policy evaluation

Given a fixed policy $\pi$, compute $V^\pi$ by **iterating the Bellman expectation
backup** until it converges:
$$V_{k+1}(s) \leftarrow \sum_a \pi(a\mid s)\sum_{s'}P(s'\mid s,a)\big[R+\gamma V_k(s')\big]$$

We represent a stochastic policy as an array `policy[s, a]` = probability of action
`a` in state `s`.

In [ ]:
def policy_evaluation(env, policy, gamma=None, theta=1e-8, max_iter=10_000):
    gamma = env.gamma if gamma is None else gamma
    V = np.zeros(env.n_states)
    for _ in range(max_iter):
        delta = 0.0
        for s in range(env.n_states):
            v_old = V[s]
            v_new = 0.0
            # TODO: apply the Bellman EXPECTATION backup for state s.
            # Sum over actions a (weighted by policy[s, a]) and over the
            # transitions in env.P[s][a] = [(prob, ns, r, done), ...].
            # Remember: a terminal next-state contributes no future value.
            raise NotImplementedError("Implement the Bellman expectation backup")
            V[s] = v_new
            delta = max(delta, abs(v_old - v_new))
        if delta < theta:
            break
    return V

random_policy = np.ones((env.n_states, env.n_actions)) / env.n_actions
V_random = policy_evaluation(env, random_policy)
plot_state_values(env, V_random, title="V of the uniform-random policy")
plt.show()

## 2. Policy improvement → Policy Iteration

Given $V^\pi$, act **greedily** with respect to the one-step lookahead
$Q(s,a)=\sum_{s'}P(s'\mid s,a)[R+\gamma V^\pi(s')]$. Alternating **evaluation** and
**greedy improvement** is **policy iteration**, and it provably converges to
$\pi^*$.

In [ ]:
def q_from_v(env, V, s, gamma):
    # TODO: return the length-n_actions vector Q(s, .) from V using env.P[s][a].
    raise NotImplementedError("Implement one-step lookahead Q(s, a)")

def policy_iteration(env, gamma=None, theta=1e-8):
    gamma = env.gamma if gamma is None else gamma
    policy = np.ones((env.n_states, env.n_actions)) / env.n_actions
    while True:
        V = policy_evaluation(env, policy, gamma, theta)
        stable = True
        for s in range(env.n_states):
            old_action = np.argmax(policy[s])
            # TODO: compute greedy action from q_from_v, make policy[s] one-hot on it,
            # and set stable=False if the greedy action changed.
            raise NotImplementedError("Implement the policy improvement step")
        if stable:
            return policy, V

pi_star, V_star = policy_iteration(env)
greedy = np.argmax(pi_star, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_state_values(env, V_star, "V* (policy iteration)", ax=axes[0])
plot_policy(env, greedy, "Optimal policy", ax=axes[1])
plt.tight_layout(); plt.show()

## 3. Value Iteration

Instead of fully evaluating each policy, **value iteration** folds improvement into
the backup by taking a `max` over actions every sweep:
$$V_{k+1}(s) \leftarrow \max_a \sum_{s'} P(s'\mid s,a)\big[R + \gamma V_k(s')\big]$$
Then extract the greedy policy once at the end. It's usually faster.

In [ ]:
def value_iteration(env, gamma=None, theta=1e-8, max_iter=10_000):
    gamma = env.gamma if gamma is None else gamma
    V = np.zeros(env.n_states)
    for it in range(max_iter):
        delta = 0.0
        for s in range(env.n_states):
            # TODO: Bellman OPTIMALITY backup: V[s] <- max_a Q(s, a).
            # Track delta = max change to test convergence.
            raise NotImplementedError("Implement the value iteration backup")
        if delta < theta:
            break
    # TODO: extract the greedy policy from the converged V
    policy = None
    raise NotImplementedError("Extract the greedy policy")
    return policy, V, it + 1

vi_policy, V_vi, n_iter = value_iteration(env)
print(f"Value iteration converged in {n_iter} sweeps")
plot_policy(env, vi_policy, "Optimal policy (value iteration)")
plt.show()

## 4. Watch the optimal policy act

DP gave us a policy — now let's *see* it. We roll out the greedy optimal policy in
the GridWorld and render it to a GIF. Because the world is slippery (`slip=0.1`),
the agent occasionally gets pushed off course but still recovers toward the goal.

In [ ]:
optimal_action = lambda s: int(vi_policy[s])
animate_policy(env, optimal_action, max_steps=40, fps=3, path="gridworld_optimal.gif")

## 5. The effect of the discount factor $\gamma$

$\gamma$ controls how far-sighted the agent is. Re-solve the environment for a few
values of $\gamma$ and see how the optimal policy changes — with a small $\gamma$
the agent is myopic and may take the shortest (riskier) path.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, g in zip(axes, [0.5, 0.9, 0.99]):
    pol, _, _ = value_iteration(env, gamma=g)
    plot_policy(env, pol, f"Optimal policy, gamma={g}", ax=ax)
plt.tight_layout(); plt.show()

## Recap
- With a known model, DP computes $V^*$ and $\pi^*$ **exactly**.
- **Policy iteration**: alternate full evaluation + greedy improvement.
- **Value iteration**: one-step max backups; often faster.
- The **discount factor** $\gamma$ shapes how far-sighted the optimal policy is.

### Going further (optional, same environment)
- Change `slip` and `step_reward` when building the GridWorld and re-solve: how does
  the optimal policy react to a riskier world?
- Compare the number of sweeps policy iteration vs. value iteration need to converge.

**Next:** Module 03 — what if we *don't* know the model? Learn from experience
with Monte Carlo and Temporal-Difference methods.